# Notebook 04 - LangGraph e RAG Medico

## Tech Challenge Fase 3 - Assistente Virtual Medico

Este notebook cobre:
1. Conceitos do LangGraph (grafos de decisao)
2. Implementacao do grafo medico com LangGraph
3. RAG (Retrieval-Augmented Generation) para consultas medicas
4. Integracao do RAG com LangGraph

---
## 1. Configuracao do Ambiente

In [ ]:
import os
import torch
from dotenv import load_dotenv
from typing import TypedDict, List, Optional
from langchain_core.documents import Document
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline

load_dotenv()

# Configuracao do Modelo Fine-Tuned
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "../models/assistente_medico_final"

print("Ambiente configurado!")
print(f"Modelo base: {MODEL_NAME}")
print(f"Adapter: {ADAPTER_PATH}")

---
## 2. Conceitos do LangGraph

### O que e LangGraph?

O LangGraph e uma biblioteca que organiza fluxos de LLMs como **grafos direcionados** em vez de pipelines lineares.

### Componentes Principais:
- **StateGraph**: Define a estrutura do grafo com TypedDict como esquema de estado
- **Nos (Nodes)**: Funcoes puras que recebem estado e retornam modificacoes
- **Arestas (Edges)**: Transicoes entre nos, incluindo condicionais
- **Estado Compartilhado**: Dicionario tipado que viaja entre os nos

### Por que usar LangGraph?
- Fluxos mais complexos que lineares
- Condicoes e ramificacoes
- Loops e iteracoes
- Paralelismo
- Controle fino do fluxo

---
## 3. Implementacao do Grafo Medico com LangGraph

### 3.1 Definicao do Estado

In [ ]:
from langgraph.graph import StateGraph, END

# 1. Definir o estado compartilhado
class MedicoState(TypedDict):
    """Estado compartilhado entre todos os nos do grafo."""
    pergunta: str                           # Pergunta do medico
    classificacao: str                      # Classificacao da consulta
    documentos_brutos: List[Document]       # Documentos recuperados
    documentos_filtrados: List[Document]    # Documentos relevantes
    contexto: str                           # Contexto para gerar resposta
    resposta: str                           # Resposta gerada
    fontes: List[str]                       # Fontes utilizadas
    confianca: float                        # Nivel de confianca (0-1)
    etapa_atual: str                        # Controle de fluxo
    historico_decisoes: List[str]           # Historico de decisoes
    alerta_necessario: bool                 # Se ha necessidade de alerta
    mensagem_alerta: str                    # Mensagem de alerta

print("Estado do grafo definido!")
print(f"Campos: {list(MedicoState.__annotations__.keys())}")

### 3.2 Implementacao dos Nos

In [ ]:
from langchain_core.prompts import PromptTemplate

# FLAG: True = usar modelo fine-tuned, False = usar modelo base puro
USE_FINETUNED = True

# AutoTokenizer: detecta automaticamente o tokenizer correto do modelo
# com base no config.json do adapter. Converte texto em numeros (tokens)
# que o LLM processa, e depois converte de volta para texto.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Carregar modelo base (usando float32 para compatibilidade com CPU)
print("Carregando modelo base...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map=None,
    trust_remote_code=True
)

# Aplicar adapter LoRA fine-tuned (opcional)
if USE_FINETUNED:
    print("Aplicando adapter LoRA...")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
else:
    print("Usando modelo BASE (sem adapter)")

model.eval()

# Criar pipeline de geracao
pipe = pipeline(
    "text-generation",    # Tipo de tarefa: gerar texto
    model=model,           # Modelo fine-tuned carregado
    tokenizer=tokenizer,   # Tokenizer do adapter
    max_length=2304,       # Max_length total (2048 contexto + 256 novos tokens)
    max_new_tokens=256,    # Maximo de tokens novos a gerar na resposta
    truncation=True,       # Trunca input se for muito longo
    temperature=0.3,       # Criatividade moderada
    do_sample=True,        # Amostragem aleatoria (necessario para temperature > 0)
    top_p=0.9,             # Nucleus sampling: ignora tokens com prob < 10%
    top_k=50,              # Top-K sampling: considera os 50 tokens mais provaveis
    repetition_penalty=1.1,# Penaliza repeticao de tokens (evita loop)
    pad_token_id=tokenizer.eos_token_id  # Token de padding = token de fim
)

# Envolver com LangChain
llm = HuggingFacePipeline(pipeline=pipe)

print(f"Modelo carregado com sucesso!")
print(f"Modo: {'Fine-tuned' if USE_FINETUNED else 'Base'}")

In [ ]:
# ============================================
# NO 1: CLASSIFICACAO DA CONSULTA
# ============================================
def classificar_consulta(state: dict) -> dict:
    """
    No de classificacao: identifica o tipo de consulta medica.
    """
    pergunta = state["pergunta"]
    
    prompt_classificacao = PromptTemplate(
        template="""### Instruction:
Classifique a seguinte consulta medica em UMA categoria:
- CLINICA_GERAL
- CARDIOLOGIA
- PNEUMOLOGIA
- NEUROLOGIA
- URGENCIA
- ENFERMAGEM

Consulta: {pergunta}

Responda APENAS com o nome da categoria.

### Response:
""",
        input_variables=["pergunta"]
    )
    
    resposta = llm.invoke(prompt_classificacao.format(pergunta=pergunta))
    classificacao = resposta.strip()
    
    historico = state.get("historico_decisoes", [])
    historico.append(f"Classificacao: {classificacao}")
    
    return {
        "classificacao": classificacao,
        "historico_decisoes": historico,
        "etapa_atual": "classificado"
    }

# ============================================
# NO 2: RECUPERACAO DE DOCUMENTOS (RAG)
# ============================================
def recuperar_documentos(state: dict) -> dict:
    """
    No de recuperacao: busca documentos relevantes na base de conhecimento.
    Em producao, isso conectaria a um vector store (FAISS, Pinecone, etc.)
    """
    classificacao = state.get("classificacao", "CLINICA_GERAL")
    
    # Simulacao de documentos recuperados
    docs_simulados = [
        Document(
            page_content=f"Protocolo de {classificacao}: Condutas padronizadas para o atendimento de pacientes na area de {classificacao}.",
            metadata={"fonte": f"Protocolo_Hospitalar_{classificacao}", "versao": "2024"}
        ),
        Document(
            page_content=f"Diretrizes clinicas para {classificacao}: Baseadas em evidencias cientificas atualizadas.",
            metadata={"fonte": f"Diretrizes_{classificacao}", "versao": "2024"}
        )
    ]
    
    historico = state.get("historico_decisoes", [])
    historico.append(f"Recuperados {len(docs_simulados)} documentos para {classificacao}")
    
    return {
        "documentos_brutos": docs_simulados,
        "historico_decisoes": historico,
        "etapa_atual": "recuperado"
    }

# ============================================
# NO 3: FILTRAGEM DE RELEVANCIA
# ============================================
def filtrar_relevancia(state: dict) -> dict:
    """
    No de filtragem: valida se documentos sao relevantes para a pergunta.
    """
    documentos = state.get("documentos_brutos", [])
    pergunta = state.get("pergunta", "")
    
    # Simulacao de filtragem (em producao, usaria similaridade semantica)
    docs_filtrados = documentos  # Por simplificacao, mantemos todos
    
    # Calcular confianca baseada na quantidade de documentos
    confianca = min(0.5 + len(docs_filtrados) * 0.15, 0.95)
    
    historico = state.get("historico_decisoes", [])
    historico.append(f"Filtrados {len(docs_filtrados)} documentos, confianca: {confianca:.2f}")
    
    return {
        "documentos_filtrados": docs_filtrados,
        "confianca": confianca,
        "historico_decisoes": historico,
        "etapa_atual": "filtrado"
    }

# ============================================
# NO 4: VALIDACAO DE QUALIDADE
# ============================================
def validar_qualidade(state: dict) -> dict:
    """
    No de validacao: verifica se a confianca e suficiente para gerar resposta.
    """
    confianca = state.get("confianca", 0)
    
    historico = state.get("historico_decisoes", [])
    
    if confianca < 0.6:
        historico.append("Validacao: confianca insuficiente, necessario reprocessar")
        return {
            "etapa_atual": "reprocessar",
            "historico_decisoes": historico
        }
    else:
        historico.append("Validacao: confianca suficiente, prosseguindo")
        return {
            "etapa_atual": "aprovado",
            "historico_decisoes": historico
        }

# ============================================
# NO 5: GERACAO DE RESPOSTA
# ============================================
def gerar_resposta(state: dict) -> dict:
    """
    No de geracao: cria a resposta medica final.
    """
    pergunta = state.get("pergunta", "")
    classificacao = state.get("classificacao", "")
    documentos = state.get("documentos_filtrados", [])
    confianca = state.get("confianca", 0)
    
    # Montar contexto dos documentos
    contexto = "\n".join([doc.page_content for doc in documentos])
    
    # Extrair fontes
    fontes = [doc.metadata.get("fonte", "Desconhecida") for doc in documentos]
    
    prompt_resposta = PromptTemplate(
        template="""### Instruction:
Voce e um assistente medico especializado.

REGRAS:
1. Nao prescreva medicamentos diretamente
2. Nao faca diagnosticos definitivos
3. SEMPRE inclua: 'Esta resposta e uma sugestao e deve ser validada por um medico'
4. SEMPRE cite as fontes utilizadas

Contexto dos protocolos:
{contexto}

Pergunta do medico: {pergunta}
Classificacao: {classificacao}
Nivel de confianca: {confianca:.0%}

Forneça uma resposta estruturada com:
1. Resumo da analise
2. Condutas recomendadas
3. Exames complementares
4. Fontes consultadas
5. Aviso de seguranca

### Response:
""",
        input_variables=["contexto", "pergunta", "classificacao", "confianca"]
    )
    
    resposta = llm.invoke(prompt_resposta.format(
        contexto=contexto,
        pergunta=pergunta,
        classificacao=classificacao,
        confianca=confianca
    ))
    
    historico = state.get("historico_decisoes", [])
    historico.append("Resposta gerada com sucesso")
    
    return {
        "resposta": resposta,
        "fontes": fontes,
        "historico_decisoes": historico,
        "etapa_atual": "respondido"
    }

# ============================================
# NO 6: VERIFICACAO DE ALERTA
# ============================================
def verificar_alerta(state: dict) -> dict:
    """
    No de alerta: verifica se e necessario um alerta urgente.
    """
    pergunta = state.get("pergunta", "").lower()
    
    # Verificar palavras-chave de urgencia
    palavras_urgencia = ["emergencia", "urgente", "parada", "infarto", "derrame", "sangramento"]
    necessita_alerta = any(palavra in pergunta for palavra in palavras_urgencia)
    
    mensagem_alerta = ""
    if necessita_alerta:
        mensagem_alerta = "ALERTA: Esta consulta indica possivel situacao de emergencia. Encaminhe imediatamente para o servico de urgencia."
    
    historico = state.get("historico_decisoes", [])
    historico.append(f"Verificacao de alerta: {'ALERTA ATIVADO' if necessita_alerta else 'Sem alerta'}")
    
    return {
        "alerta_necessario": necessita_alerta,
        "mensagem_alerta": mensagem_alerta,
        "historico_decisoes": historico,
        "etapa_atual": "finalizado"
    }

print("Todos os nos implementados!")
print("Nos: classificar_consulta, recuperar_documentos, filtrar_relevancia, validar_qualidade, gerar_resposta, verificar_alerta")

### 3.3 Construcao do Grafo

In [ ]:
# ============================================
# CONSTRUCAO DO GRAFO LANGGRAPH
# ============================================
# Fluxo: Classificar -> Recuperar -> Filtrar -> Validar -> Responder -> Alertar
# Se validacao falhar: volta para Recuperar (loop)

workflow = StateGraph(MedicoState)

# Adicionar nos do grafo
workflow.add_node("classificar", classificar_consulta)   # No 1
workflow.add_node("recuperar", recuperar_documentos)     # No 2
workflow.add_node("filtrar", filtrar_relevancia)         # No 3
workflow.add_node("validar", validar_qualidade)          # No 4
workflow.add_node("responder", gerar_resposta)           # No 5
workflow.add_node("alertar", verificar_alerta)           # No 6

# Definir ponto de entrada
workflow.set_entry_point("classificar")

# Arestas sequenciais (fluxo normal)
workflow.add_edge("classificar", "recuperar")
workflow.add_edge("recuperar", "filtrar")
workflow.add_edge("filtrar", "validar")

# Aresta condicional: se confianca >= 0.6 vai para responder, senao volta para recuperar
workflow.add_conditional_edges(
    "validar",
    lambda state: state["etapa_atual"],
    {
        "aprovado": "responder",
        "reprocessar": "recuperar",  # Loop: tenta recuperar mais documentos
    }
)

# Apos responder, verifica se ha alerta de urgencia
workflow.add_edge("responder", "alertar")

# Ponto final
workflow.add_edge("alertar", END)

# Compilar o grafo
app = workflow.compile()

print("Grafo compilado com sucesso!")

### 3.4 Visualizacao do Grafo

In [ ]:
# Visualizar estrutura do grafo
print("Estrutura do grafo:")
try:
    print(app.get_graph().draw_ascii())
except ImportError:
    print("Para visualizar o grafo, instale: pip install grandalf")
    print("\nNos do grafo:")
    for node in app.get_graph().nodes:
        print(f"  - {node}")

### 3.5 Execucao do Grafo

In [ ]:
# Executar o grafo com uma pergunta de teste
print("="*60)
print("EXECUCAO DO GRAFO LANGGRAPH")
print("="*60)

entrada = {
    "pergunta": "Qual o protocolo para pneumonia hospitalar em pacientes idosos?",
    "historico_decisoes": []
}

resultado = app.invoke(entrada)

print(f"\nClassificacao: {resultado['classificacao']}")
print(f"Confianca: {resultado['confianca']:.2%}")
print(f"Fontes: {resultado['fontes']}")
print(f"Alerta necessario: {resultado['alerta_necessario']}")
if resultado['alerta_necessario']:
    print(f"Mensagem alerta: {resultado['mensagem_alerta']}")
print(f"\nHistorico de decisoes:")
for i, decisao in enumerate(resultado['historico_decisoes'], 1):
    print(f"  {i}. {decisao}")
print(f"\n{'='*60}")
print("RESPOSTA DO ASSISTENTE:")
print("="*60)
print(resultado['resposta'])

---
## 4. RAG (Retrieval-Augmented Generation)

### 4.1 Conceito

RAG combina:
- **Retrieval** (Recuperacao): Busca de documentos relevantes em uma base de conhecimento
- **Augmented** (Aumentado): Enriquece o prompt do LLM com os documentos recuperados
- **Generation** (Geracao): O LLM gera resposta baseada no contexto recuperado

### Vantagens do RAG:
- Reduz alucinacoes do LLM
- Permite acesso a informacoes atualizadas
- Fontes rastreaveis e auditaveis
- Conhecimento especifico do dominio

### 4.2 Vector Store com FAISS

In [ ]:
# Importar modulo RAG do projeto
import sys
sys.path.append('..')
from src.rag_module import (
    SimpleDeterministicEmbeddings,
    create_medical_documents,
    create_vector_store,
    create_retriever
)

# Criar documentos medicos
documentos_medicos = create_medical_documents()
print(f"{len(documentos_medicos)} documentos medicos carregados")

# Criar vector store
print("Criando vector store com FAISS...")
vectorstore = create_vector_store(documentos_medicos)
print(f"Vector store criado com {vectorstore.index.ntotal} vetores")

# Criar retriever
retriever = create_retriever(vectorstore, search_k=2)
print("Retriever configurado!")

### 4.3 Integracao RAG com LangGraph

In [ ]:
# Atualizar no de recuperacao para usar RAG real
def recuperar_documentos_rag(state: dict) -> dict:
    """
    No de recuperacao com RAG: busca documentos no vector store.
    """
    pergunta = state["pergunta"]
    
    # Buscar documentos relevantes
    documentos = retriever.invoke(pergunta)
    
    historico = state.get("historico_decisoes", [])
    historico.append(f"RAG: Recuperados {len(documentos)} documentos para a pergunta")
    
    return {
        "documentos_brutos": documentos,
        "historico_decisoes": historico,
        "etapa_atual": "recuperado"
    }

# Construir novo grafo com RAG
workflow_rag = StateGraph(MedicoState)

workflow_rag.add_node("classificar", classificar_consulta)
workflow_rag.add_node("recuperar_rag", recuperar_documentos_rag)
workflow_rag.add_node("filtrar", filtrar_relevancia)
workflow_rag.add_node("validar", validar_qualidade)
workflow_rag.add_node("responder", gerar_resposta)
workflow_rag.add_node("alertar", verificar_alerta)

workflow_rag.set_entry_point("classificar")
workflow_rag.add_edge("classificar", "recuperar_rag")
workflow_rag.add_edge("recuperar_rag", "filtrar")
workflow_rag.add_edge("filtrar", "validar")

workflow_rag.add_conditional_edges(
    "validar",
    lambda state: state["etapa_atual"],
    {
        "aprovado": "responder",
        "reprocessar": "recuperar_rag",
    }
)

workflow_rag.add_edge("responder", "alertar")
workflow_rag.add_edge("alertar", END)

app_rag = workflow_rag.compile()

print("Grafo RAG compilado com sucesso!")

In [ ]:
# Testar o grafo RAG
print("="*60)
print("TESTE DO GRAFO RAG")
print("="*60)

entrada_rag = {
    "pergunta": "Qual o tratamento para sepse em pacientes criticos?",
    "historico_decisoes": []
}

resultado_rag = app_rag.invoke(entrada_rag)

print(f"\nClassificacao: {resultado_rag['classificacao']}")
print(f"Confianca: {resultado_rag['confianca']:.2%}")
print(f"Fontes: {resultado_rag['fontes']}")
print(f"\nHistorico:")
for i, decisao in enumerate(resultado_rag['historico_decisoes'], 1):
    print(f"  {i}. {decisao}")
print(f"\n{'='*60}")
print("RESPOSTA:")
print("="*60)
print(resultado_rag['resposta'])

---
## 5. Padrao ReAct (Reasoning + Acting)

O padrao ReAct e utilizado em agentes que combinam raciocinio com acoes.

In [ ]:
# Exemplo de padrao ReAct
def agente_react_medico(pergunta: str) -> str:
    """
    Agente medico seguindo o padrao ReAct.
    """
    # Thought: Raciocinio
    thought = f"""
    Thought: Preciso analisar a pergunta medica:
    1. Identificar a condicao clinica
    2. Buscar protocolo relevante
    3. Verificar se ha contraindicacoes
    4. Gerar resposta segura
    """
    
    # Action: Buscar protocolo
    documentos_encontrados = retriever.invoke(pergunta)
    
    # Observation: Processar resultados
    if documentos_encontrados:
        obs = f"Observation: Encontrei {len(documentos_encontrados)} protocolos relevantes."
        contexto = "\n".join([doc.page_content for doc in documentos_encontrados])
    else:
        obs = "Observation: Nenhum protocolo especifico encontrado. Usando conhecimento geral."
        contexto = ""
    
    # Gerar resposta final
    prompt_final = f"""### Instruction:
Pergunta: {pergunta}
Contexto: {contexto}

Gere uma resposta medica segura e estruturada.
Inclua aviso de seguranca.

### Response:
"""
    
    resposta = llm.invoke(prompt_final)
    
    return f"{thought}\n{obs}\n\nResposta:\n{resposta}"

# Testar agente ReAct
print("="*60)
print("AGENTE REACT MEDICO")
print("="*60)
resultado_react = agente_react_medico("Paciente com insuficiencia cardiaca aguda. Qual conduta?")
print(resultado_react)

---
## 6. Resumo

### Componentes Implementados:

| Componente | Funcao |
|------------|--------|
| **StateGraph** | Estrutura do grafo |
| **TypedDict** | Estado compartilhado |
| **Nos** | classificar, recuperar, filtrar, validar, responder, alertar |
| **Arestas Condicionais** | Fluxo baseado em validacao |
| **FAISS** | Vector store para busca semantica |
| **RAG** | Recuperacao + Geracao |
| **ReAct** | Padrao Reasoning + Acting |

### Fluxo do Assistente Medico:
```
Pergunta -> Classificacao -> Recuperacao (RAG) -> Filtragem -> 
Validacao -> (se aprovado) Resposta -> Alerta -> Fim
                     (se reprocessar) ^
```

### Proximo notebook:
O notebook `05_sistema_completo.ipynb` integrara tudo com seguranca e logging detalhado.

In [ ]:
print("\n=== NOTEBOOK 04 CONCLUIDO ===")
print("LangGraph e RAG implementados com sucesso!")